### Import Dependencies

In [1]:
import os
import openai
from qdrant_client import QdrantClient
from langsmith import Client

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

### Download an example reference data point from LangSmith

In [2]:
from dotenv import load_dotenv
import os

load_dotenv("../../.env")

True

In [3]:
ls_client = Client()

In [4]:
dataset = ls_client.read_dataset(
    dataset_name="coordinator-delegation-evaluation"
)

In [5]:
dataset

Dataset(name='coordinator-delegation-evaluation', description='', data_type=<DataType.kv: 'kv'>, id=UUID('b1ee8607-e6a2-4c1d-ab26-12032e443752'), created_at=datetime.datetime(2026, 9, 2, 21, 23, 11, 600301, tzinfo=TzInfo(0)), modified_at=datetime.datetime(2026, 9, 2, 21, 23, 11, 600301, tzinfo=TzInfo(0)), example_count=3, session_count=0, last_session_start_time=None, inputs_schema=None, outputs_schema=None, transformations=None, metadata=None)

In [6]:
reference_inputs = [item.inputs["input"] for item in list(ls_client.list_examples(dataset_id=dataset.id, limit=50))]

In [7]:
reference_inputs

[{'messages': [{'type': 'human',
    'content': 'Can I get a green speaker for outdoors use?',
    'additional_kwargs': {},
    'response_metadata': {}},
   {'id': 'resp_04dd61ec7b771631006a99a6be6e4887d1875207350908b2c9',
    'type': 'ai',
    'content': [{'id': 'rs_04dd61ec7b771631006a99a6bf19d887d1b0d9f67deb7c3b43',
      'type': 'reasoning',
      'content': [],
      'summary': [],
      'encrypted_content': 'gAAAAABqmaa_CfBIfr3pCI-NtyavdSWdwGFMZvqa-ej3WvYiySk5wLY1DP-6df0qJaQQ3BvZJR1NgVrCRl39KSjw-dIHCdHHZ533iiMxqtkynUHDMbOMmAx8oTa3Gux-koNKeZXs94J-xGovPfERiKq5PrJLmiZNKFy3NB7lXcSImiQFFKnvWnrBEJcY_Na4EE7BtoF-3EyNaJo8xJ9XbfuJoX8bRzRKrB3VDg8JTxQDAYOz4622sSVPHyqfRU07emLs9by1LDAEPdKJzFGs7D2q2WFBNncwANRB5Pu1Y84kCmtx9FkWRtjjALkr7gFPsArBvyQrCwUoSDTNbsOmdvUKjHjS5SVA6CkNGq8B4_88dXLDISV80xqs2M-EaiUp83WFI6BplA3uPgIH-m9BnH6ZnN5GvvPUYwZuRkAMgz4LUhW8RAs4pudiR9AgLVMWAZgFzDnJsQvdFJ-nIMqzNiqP26WEz5v-pqDTRRXYTZx6dCovGCMv86fcrZifvU018yj75svz-mS7EpZsC4Dil-XT_I6XgrG8vlhe6tcpmS5RU2-E0174i7LDKD5x41rjJGs7jD

In [8]:
reference_outputs = [item.outputs for item in list(ls_client.list_examples(dataset_id=dataset.id, limit=50))]

In [9]:
reference_outputs

[{'answer': '',
  'coordinator_agent': {'plan': [], 'next_agent': '', 'final_answer': True}},
 {'answer': '',
  'coordinator_agent': {'plan': [], 'next_agent': '', 'final_answer': True}},
 {'answer': '',
  'coordinator_agent': {'plan': [{'task': 'Reserve the selected earphones and laptop items in the warehouses.',
     'agent': 'warehouse_manager_agent'}],
   'next_agent': 'warehouse_manager_agent',
   'final_answer': False}}]

### Coordinator Agent Evaluation

In [10]:
from pydantic import BaseModel

from langchain_openai import ChatOpenAI

from langsmith import traceable

from langchain_core.messages import SystemMessage, AIMessage
from IPython.display import display

from typing import Any, Annotated, List
from pydantic import Field
from operator import add

from jinja2 import Template

In [11]:
class Delegation(BaseModel):
    agent: str = Field(description="The agent to delegate the task to.")
    task: str = Field(description="The task to be performed by the agent.")

class Plan(BaseModel):
    next_agent: str = Field(description="The next agent to invoke")
    plan: List[Delegation] = Field(description="A list of delegations to agents with tasks to be performed in sequence.")

class FinalAgentResponse(BaseModel):

    answer: str = Field(description="Answer to the question")

class CoordinatorAgentProperties(BaseModel):
    iteration: int = 0
    final_answer: bool = False
    plan: List[Delegation] = []
    next_agent: str = ""
    
class State(BaseModel):
    messages: Annotated[List[Any], add] = []
    coordinator_agent: CoordinatorAgentProperties = CoordinatorAgentProperties()
    answer: str = ""

In [12]:
@traceable(
    name="coordinator_agent",
    run_type="llm",
    metadata={
        "ls_provider": "openai",
        "ls_model_name": "gpt-5.4-mini"
    }
)
def coordinator_agent(state) -> dict:
    
    prompt_template = """You are a Coordinator Agent as part of a shopping assistant.

## Instructions

- Your role is to create plans for solving user queries and delegate the tasks accordingly.
- You will be given a conversation history, your task is to create a plan for solving the user's query.
- After the plan is created, you should output the next agent to invoke and the task to be performed by that agent.
- Once an agent finishes its task, you will be handed the control back, you should then review the conversation history and revise the plan.
- If there is a sequence of tasks to be performed by a single agent, you should combine them into a single task.
- Do not route to any agent if the user's query needs clarification or is irelevant. Do it yourself.

## Available Agents

- product_qna_agent: The user is asking a question about a product. This can be a question about available products, their specifications, user reviews etc.
- shopping_cart_agent: The user is asking to add or remove items from the shopping cart or questions about the current shopping cart.
- warehouse_manager_agent: The user is asking to reserve items from the warehouses or about availability of the items in warehouses.

## Examples

Question: "Do you have running shoes under $100?"
Next agent: product_qna_agent

Question: "Can you list the items in my cart?"
Next agent: shopping_cart_agent

Question: "Can you reserve my shopping cart?"
Next agent: warehouse_manager_agent
"""

    template = Template(prompt_template)

    prompt = template.render()

    llm = ChatOpenAI(
        model="gpt-5.4-mini",
        reasoning_effort="low",
        use_responses_api=True
    )
    llm_with_tools = llm.bind_tools(
        [FinalAgentResponse, Plan],
        tool_choice="required"
    )

    response = llm_with_tools.invoke(
        [
            SystemMessage(content=prompt),
            *state.messages
        ]
    )

    final_answer = False
    answer = ""
    plan = []
    next_agent = ""

    def sanitise_response(response):

        for tool_call in response.tool_calls:
            if tool_call.get("name") == "FinalAgentResponse":
                answer = tool_call.get("args").get("answer")

        return AIMessage(content=answer)

    if len(response.tool_calls) > 0:
        if response.tool_calls[0].get("name") == "Plan":
            plan = response.tool_calls[0].get("args").get("plan")
            next_agent = response.tool_calls[0].get("args").get("next_agent")
            response = None
        else:
            for tool_call in response.tool_calls:
                if tool_call.get("name") == "FinalAgentResponse":
                    final_answer = True
                    answer = tool_call.get("args").get("answer")

                    response = sanitise_response(response)

    return {
        "messages": [response] if response else [],
        "coordinator_agent": {
            "iteration": state.coordinator_agent.iteration + 1,
            "final_answer": final_answer,
            "plan": plan,
            "next_agent": next_agent
        },
        "answer": answer
    }

In [13]:
output = coordinator_agent(
    State(
        messages=reference_inputs[0]["messages"],
        coordinator_agent=CoordinatorAgentProperties(
            iteration=0,
            final_answer=False,
            plan=[],
            next_agent=""
        ),
        answer=""
    )
)

In [14]:
output

{'messages': [],
 'coordinator_agent': {'iteration': 1,
  'final_answer': False,
  'plan': [{'agent': 'product_qna_agent',
    'task': 'Find whether there are any green speakers suitable for outdoor use, and if so provide matching product options with key outdoor-related specs such as waterproofing, portability, battery life, and color. If no exact match exists, provide the closest outdoor speaker alternatives and note the color mismatch.'}],
  'next_agent': 'product_qna_agent'},
 'answer': ''}

In [15]:
def evaluate_coordinator_delegation(run, example):

    final_answer_match = run["coordinator_agent"]["final_answer"] == example["coordinator_agent"]["final_answer"]
    next_agent_match = run["coordinator_agent"]["next_agent"] == example["coordinator_agent"]["next_agent"]

    return final_answer_match and next_agent_match

In [16]:
evaluate_coordinator_delegation(output, reference_outputs[0])

False

### Run against LangSmith

In [17]:
def evaluate_coordinator_delegation(run, example):

    final_answer_match = run.outputs["coordinator_agent"]["final_answer"] == example.outputs["coordinator_agent"]["final_answer"]
    next_agent_match = run.outputs["coordinator_agent"]["next_agent"] == example.outputs["coordinator_agent"]["next_agent"]

    return final_answer_match and next_agent_match

In [ ]:
results_hybrid = ls_client.evaluate(
    lambda x: coordinator_agent(
        State(
            messages=x["input"]["messages"],
            coordinator_agent=CoordinatorAgentProperties(
                iteration=0,
                final_answer=False,
                plan=[],
                next_agent=""
            ),
            answer=""
    )),
    data="coordinator-delegation-evaluation",
    evaluators=[
        evaluate_coordinator_delegation
    ],
    experiment_prefix="coordinator-delegation",
    max_concurrency=10, # 10 runs in parallel
    num_repetitions=5
)


### Coordinator Agegnt V1 (update prompt)

In [20]:
class Delegation(BaseModel):
    agent: str = Field(description="The agent to delegate the task to.")
    task: str = Field(description="The task to be performed by the agent.")

class Plan(BaseModel):
    next_agent: str = Field(description="The next agent to invoke")
    plan: List[Delegation] = Field(description="A list of delegations to agents with tasks to be performed in sequence.")

class FinalAgentResponse(BaseModel):

    answer: str = Field(description="Answer to the question")

class CoordinatorAgentProperties(BaseModel):
    iteration: int = 0
    final_answer: bool = False
    plan: List[Delegation] = []
    next_agent: str = ""
    
class State(BaseModel):
    messages: Annotated[List[Any], add] = []
    coordinator_agent: CoordinatorAgentProperties = CoordinatorAgentProperties()
    answer: str = ""

In [ ]:
@traceable(
    name="coordinator_agent",
    run_type="llm",
    metadata={
        "ls_provider": "openai",
        "ls_model_name": "gpt-5.4-mini"
    }
)
def coordinator_agent(state) -> dict:
    
    prompt_template = """You are a Coordinator Agent as part of a shopping assistant.

## Instructions

- Your role is to delegate work to worker agents in order to solve user queries.
- You should output the next agent to invoke.
- Once an agent finishes its task, you will be handed the control back, you should then review the conversation history and output the next delegation via the `Plan` tool or output the final answer via the `FinalAgentResponse` tool.
- Do not route to any agent if the user's query needs clarification or is irelevant. Do it yourself.
- Only route to the `warehouse_manager_agent` if the user has confirmed that they want to reserve an item and it was already successfully added to the shopping cart.
- Only route to the `shopping_cart_agent` if the user specifically asks about their shopping cart or wants to add or remove items from the shopping cart.
- Do not delegate work to the same agent twice in a row.

## Routing rules

Match the user's latest message to one of these patterns:

1. PRODUCT QUESTION — asks about products, specs, reviews, availability, suggestions.
    Examples: "can I get some earphones", "show me laptops", "what watches do you have", "tell me about X", "is X any good", "I'd like to see some X".
    → Route ONLY to `product_qna_agent`. When it returns, emit `FinalAgentResponse`. Do NOT chain to cart or warehouse.

2. EXPLICIT CART INSTRUCTION — uses cart vocabulary.
    Examples: "add to my cart", "remove from my cart", "what's in my cart", "put X in my cart", "delete X from cart".
    → Route to `shopping_cart_agent`. (If items must be discovered first, route to `product_qna_agent` first.)

3. EXPLICIT RESERVATION INSTRUCTION — uses warehouse/reservation vocabulary.
    Examples: "reserve these", "hold these for me", "check warehouse stock for X".
    → Route to `warehouse_manager_agent` ONLY AFTER items are confirmed in the cart.

Default rule: if the user's message does NOT contain explicit cart or warehouse vocabulary, do NOT delegate to those agents. Phrases like "can I get", "I want",
"I'd like", "show me", "I need" are PRODUCT QUESTIONS, not purchase instructions. The user must explicitly say "add", "cart", "reserve", "buy", or "order" before those agents are in scope.

## Available Agents

- product_qna_agent: The user is asking a question about a product. This can be a question about available products, their specifications, user reviews etc.
- shopping_cart_agent: The user is asking to add or remove items from the shopping cart or questions about the current shopping cart.
- warehouse_manager_agent: The user is asking to reserve items from the warehouses or about availability of the items in warehouses.
"""

    template = Template(prompt_template)

    prompt = template.render()

    llm = ChatOpenAI(
        model="gpt-5.4-mini",
        reasoning_effort="low",
        use_responses_api=True
    )
    llm_with_tools = llm.bind_tools(
        [FinalAgentResponse, Plan],
        tool_choice="required"
    )

    response = llm_with_tools.invoke(
        [
            SystemMessage(content=prompt),
            *state.messages
        ]
    )

    final_answer = False
    answer = ""
    plan = []
    next_agent = ""

    def sanitise_response(response):

        for tool_call in response.tool_calls:
            if tool_call.get("name") == "FinalAgentResponse":
                answer = tool_call.get("args").get("answer")

        return AIMessage(content=answer)

    if len(response.tool_calls) > 0:
        if response.tool_calls[0].get("name") == "Plan":
            plan = response.tool_calls[0].get("args").get("plan")
            next_agent = response.tool_calls[0].get("args").get("next_agent")
            response = None
        else:
            for tool_call in response.tool_calls:
                if tool_call.get("name") == "FinalAgentResponse":
                    final_answer = True
                    answer = tool_call.get("args").get("answer")

                    response = sanitise_response(response)

    return {
        "messages": [response] if response else [],
        "coordinator_agent": {
            "iteration": state.coordinator_agent.iteration + 1,
            "final_answer": final_answer,
            "plan": plan,
            "next_agent": next_agent
        },
        "answer": answer
    }

In [ ]:
results = ls_client.evaluate(
    lambda x: coordinator_agent(
        State(
            messages=x["input"]["messages"],
            coordinator_agent=CoordinatorAgentProperties(
                iteration=0,
                final_answer=False,
                plan=[],
                next_agent=""
            ),
            answer=""
        )
    ),
    data="coordinator-delegation-evaluation",
    evaluators=[
        evaluate_coordinator_delegation
    ],
    experiment_prefix="coordinator-delegation",
    max_concurrency=10,
    num_repetitions=5
)

### Coordinator Agegnt V2 (Improvised prompt)

In [21]:
class Delegation(BaseModel):
    agent: str = Field(description="The agent to delegate the task to.")
    task: str = Field(description="The task to be performed by the agent.")

class Plan(BaseModel):
    next_agent: str = Field(description="The next agent to invoke")
    plan: List[Delegation] = Field(description="A list of delegations to agents with tasks to be performed in sequence.")

class FinalAgentResponse(BaseModel):

    answer: str = Field(description="Answer to the question")

class CoordinatorAgentProperties(BaseModel):
    iteration: int = 0
    final_answer: bool = False
    plan: List[Delegation] = []
    next_agent: str = ""
    
class State(BaseModel):
    messages: Annotated[List[Any], add] = []
    coordinator_agent: CoordinatorAgentProperties = CoordinatorAgentProperties()
    answer: str = ""

In [22]:
@traceable(
    name="coordinator_agent",
    run_type="llm",
    metadata={
        "ls_provider": "openai",
        "ls_model_name": "gpt-5.4-mini"
    }
)
def coordinator_agent(state) -> dict:
    
    prompt_template = """You are a Coordinator Agent as part of a shopping assistant.

## Instructions

- Your role is to delegate work to worker agents in order to solve user queries.
- You should output the next agent to invoke.
- Once an agent finishes its task, you will be handed the control back, you should then review the conversation history and output the next delegation via the `Plan` tool.
- If you do not need to delegate work to any agent, output the final answer via the `FinalAgentResponse` tool.
- Do not route to any agent if the user's query needs clarification or is irelevant. Do it yourself.
- Only route to the `warehouse_manager_agent` if the user has confirmed that they want to reserve an item and it was already successfully added to the shopping cart.
- Only route to the `shopping_cart_agent` if the user specifically asks about their shopping cart or wants to add or remove items from the shopping cart.
- Do not delegate work to the same agent twice in a row.

## Routing rules

Match the user's latest message to one of these patterns:

1. PRODUCT QUESTION — asks about products, specs, reviews, availability, suggestions.
    Examples: "can I get some earphones", "show me laptops", "what watches do you have", "tell me about X", "is X any good", "I'd like to see some X".
    → Route ONLY to `product_qna_agent`. When it returns, emit `FinalAgentResponse`. Do NOT chain to cart or warehouse.

2. EXPLICIT CART INSTRUCTION — uses cart vocabulary.
    Examples: "add to my cart", "remove from my cart", "what's in my cart", "put X in my cart", "delete X from cart".
    → Route to `shopping_cart_agent`. (If items must be discovered first, route to `product_qna_agent` first.)

3. EXPLICIT RESERVATION INSTRUCTION — uses warehouse/reservation vocabulary.
    Examples: "reserve these", "hold these for me", "check warehouse stock for X".
    → Route to `warehouse_manager_agent` ONLY AFTER items are confirmed in the cart.

Default rule: if the user's message does NOT contain explicit cart or warehouse vocabulary, do NOT delegate to those agents. Phrases like "can I get", "I want",
"I'd like", "show me", "I need" are PRODUCT QUESTIONS, not purchase instructions. The user must explicitly say "add", "cart", "reserve", "buy", or "order" before those agents are in scope.

## Available Agents

- product_qna_agent: The user is asking a question about a product. This can be a question about available products, their specifications, user reviews etc.
- shopping_cart_agent: The user is asking to add or remove items from the shopping cart or questions about the current shopping cart.
- warehouse_manager_agent: The user is asking to reserve items from the warehouses or about availability of the items in warehouses.
"""

    template = Template(prompt_template)

    prompt = template.render()

    llm = ChatOpenAI(
        model="gpt-5.4-mini",
        reasoning_effort="low",
        use_responses_api=True
    )
    llm_with_tools = llm.bind_tools(
        [FinalAgentResponse, Plan],
        tool_choice="required"
    )

    response = llm_with_tools.invoke(
        [
            SystemMessage(content=prompt),
            *state.messages
        ]
    )

    final_answer = False
    answer = ""
    plan = []
    next_agent = ""

    def sanitise_response(response):

        for tool_call in response.tool_calls:
            if tool_call.get("name") == "FinalAgentResponse":
                answer = tool_call.get("args").get("answer")

        return AIMessage(content=answer)

    if len(response.tool_calls) > 0:
        if response.tool_calls[0].get("name") == "Plan":
            plan = response.tool_calls[0].get("args").get("plan")
            next_agent = response.tool_calls[0].get("args").get("next_agent")
            response = None
        else:
            for tool_call in response.tool_calls:
                if tool_call.get("name") == "FinalAgentResponse":
                    final_answer = True
                    answer = tool_call.get("args").get("answer")

                    response = sanitise_response(response)

    return {
        "messages": [response] if response else [],
        "coordinator_agent": {
            "iteration": state.coordinator_agent.iteration + 1,
            "final_answer": final_answer,
            "plan": plan,
            "next_agent": next_agent
        },
        "answer": answer
    }

In [ ]:
results = ls_client.evaluate(
    lambda x: coordinator_agent(
        State(
            messages=x["input"]["messages"],
            coordinator_agent=CoordinatorAgentProperties(
                iteration=0,
                final_answer=False,
                plan=[],
                next_agent=""
            ),
            answer=""
        )
    ),
    data="coordinator-delegation-evaluation",
    evaluators=[
        evaluate_coordinator_delegation
    ],
    experiment_prefix="coordinator-delegation",
    max_concurrency=10,
    num_repetitions=5
)

### Coordinator Agegnt V3 (reasoning_effort="medium")

In [ ]:
class Delegation(BaseModel):
    agent: str = Field(description="The agent to delegate the task to.")
    task: str = Field(description="The task to be performed by the agent.")

class Plan(BaseModel):
    next_agent: str = Field(description="The next agent to invoke")
    plan: List[Delegation] = Field(description="A list of delegations to agents with tasks to be performed in sequence.")

class FinalAgentResponse(BaseModel):

    answer: str = Field(description="Answer to the question")

class CoordinatorAgentProperties(BaseModel):
    iteration: int = 0
    final_answer: bool = False
    plan: List[Delegation] = []
    next_agent: str = ""
    
class State(BaseModel):
    messages: Annotated[List[Any], add] = []
    coordinator_agent: CoordinatorAgentProperties = CoordinatorAgentProperties()
    answer: str = ""

In [ ]:
@traceable(
    name="coordinator_agent",
    run_type="llm",
    metadata={
        "ls_provider": "openai",
        "ls_model_name": "gpt-5.4-mini"
    }
)
def coordinator_agent(state) -> dict:
    
    prompt_template = """You are a Coordinator Agent as part of a shopping assistant.

## Instructions

- Your role is to delegate work to worker agents in order to solve user queries.
- You should output the next agent to invoke.
- Once an agent finishes its task, you will be handed the control back, you should then review the conversation history and output the next delegation via the `Plan` tool.
- If you do not need to delegate work to any agent, output the final answer via the `FinalAgentResponse` tool.
- Do not route to any agent if the user's query needs clarification or is irelevant. Do it yourself.
- Only route to the `warehouse_manager_agent` if the user has confirmed that they want to reserve an item and it was already successfully added to the shopping cart.
- Only route to the `shopping_cart_agent` if the user specifically asks about their shopping cart or wants to add or remove items from the shopping cart.
- Do not delegate work to the same agent twice in a row.

## Routing rules

Match the user's latest message to one of these patterns:

1. PRODUCT QUESTION — asks about products, specs, reviews, availability, suggestions.
    Examples: "can I get some earphones", "show me laptops", "what watches do you have", "tell me about X", "is X any good", "I'd like to see some X".
    → Route ONLY to `product_qna_agent`. When it returns, emit `FinalAgentResponse`. Do NOT chain to cart or warehouse.

2. EXPLICIT CART INSTRUCTION — uses cart vocabulary.
    Examples: "add to my cart", "remove from my cart", "what's in my cart", "put X in my cart", "delete X from cart".
    → Route to `shopping_cart_agent`. (If items must be discovered first, route to `product_qna_agent` first.)

3. EXPLICIT RESERVATION INSTRUCTION — uses warehouse/reservation vocabulary.
    Examples: "reserve these", "hold these for me", "check warehouse stock for X".
    → Route to `warehouse_manager_agent` ONLY AFTER items are confirmed in the cart.

Default rule: if the user's message does NOT contain explicit cart or warehouse vocabulary, do NOT delegate to those agents. Phrases like "can I get", "I want",
"I'd like", "show me", "I need" are PRODUCT QUESTIONS, not purchase instructions. The user must explicitly say "add", "cart", "reserve", "buy", or "order" before those agents are in scope.

## Available Agents

- product_qna_agent: The user is asking a question about a product. This can be a question about available products, their specifications, user reviews etc.
- shopping_cart_agent: The user is asking to add or remove items from the shopping cart or questions about the current shopping cart.
- warehouse_manager_agent: The user is asking to reserve items from the warehouses or about availability of the items in warehouses.
"""

    template = Template(prompt_template)

    prompt = template.render()

    llm = ChatOpenAI(
        model="gpt-5.4-mini",
        reasoning_effort="medium",
        use_responses_api=True
    )
    llm_with_tools = llm.bind_tools(
        [FinalAgentResponse, Plan],
        tool_choice="required"
    )

    response = llm_with_tools.invoke(
        [
            SystemMessage(content=prompt),
            *state.messages
        ]
    )

    final_answer = False
    answer = ""
    plan = []
    next_agent = ""

    def sanitise_response(response):

        for tool_call in response.tool_calls:
            if tool_call.get("name") == "FinalAgentResponse":
                answer = tool_call.get("args").get("answer")

        return AIMessage(content=answer)

    if len(response.tool_calls) > 0:
        if response.tool_calls[0].get("name") == "Plan":
            plan = response.tool_calls[0].get("args").get("plan")
            next_agent = response.tool_calls[0].get("args").get("next_agent")
            response = None
        else:
            for tool_call in response.tool_calls:
                if tool_call.get("name") == "FinalAgentResponse":
                    final_answer = True
                    answer = tool_call.get("args").get("answer")

                    response = sanitise_response(response)

    return {
        "messages": [response] if response else [],
        "coordinator_agent": {
            "iteration": state.coordinator_agent.iteration + 1,
            "final_answer": final_answer,
            "plan": plan,
            "next_agent": next_agent
        },
        "answer": answer
    }

In [ ]:
results = ls_client.evaluate(
    lambda x: coordinator_agent(
        State(
            messages=x["input"]["messages"],
            coordinator_agent=CoordinatorAgentProperties(
                iteration=0,
                final_answer=False,
                plan=[],
                next_agent=""
            ),
            answer=""
        )
    ),
    data="coordinator-delegation-evaluation",
    evaluators=[
        evaluate_coordinator_delegation
    ],
    experiment_prefix="coordinator-delegation",
    max_concurrency=10,
    num_repetitions=5
)

### Coordinator Agegnt V4 (comment plan), might be forcing agent to give next delegation

In [ ]:
class Delegation(BaseModel):
    agent: str = Field(description="The agent to delegate the task to.")
    task: str = Field(description="The task to be performed by the agent.")

class Plan(BaseModel):
    next_agent: str = Field(description="The next agent to invoke")
    # plan: List[Delegation] = Field(description="A list of delegations to agents with tasks to be performed in sequence.")

class FinalAgentResponse(BaseModel):

    answer: str = Field(description="Answer to the question")

class CoordinatorAgentProperties(BaseModel):
    iteration: int = 0
    final_answer: bool = False
    # plan: List[Delegation] = []
    next_agent: str = ""
    
class State(BaseModel):
    messages: Annotated[List[Any], add] = []
    coordinator_agent: CoordinatorAgentProperties = CoordinatorAgentProperties()
    answer: str = ""

In [ ]:
@traceable(
    name="coordinator_agent",
    run_type="llm",
    metadata={
        "ls_provider": "openai",
        "ls_model_name": "gpt-5.4-mini"
    }
)
def coordinator_agent(state) -> dict:
    
    prompt_template = """You are a Coordinator Agent as part of a shopping assistant.

## Instructions

- Your role is to delegate work to worker agents in order to solve user queries.
- You should output the next agent to invoke.
- Once an agent finishes its task, you will be handed the control back, you should then review the conversation history and output the next delegation via the `Plan` tool.
- If you do not need to delegate work to any agent, output the final answer via the `FinalAgentResponse` tool.
- Do not route to any agent if the user's query needs clarification or is irelevant. Do it yourself.
- Only route to the `warehouse_manager_agent` if the user has confirmed that they want to reserve an item and it was already successfully added to the shopping cart.
- Only route to the `shopping_cart_agent` if the user specifically asks about their shopping cart or wants to add or remove items from the shopping cart.
- Do not delegate work to the same agent twice in a row.

## Routing rules

Match the user's latest message to one of these patterns:

1. PRODUCT QUESTION — asks about products, specs, reviews, availability, suggestions.
    Examples: "can I get some earphones", "show me laptops", "what watches do you have", "tell me about X", "is X any good", "I'd like to see some X".
    → Route ONLY to `product_qna_agent`. When it returns, emit `FinalAgentResponse`. Do NOT chain to cart or warehouse.

2. EXPLICIT CART INSTRUCTION — uses cart vocabulary.
    Examples: "add to my cart", "remove from my cart", "what's in my cart", "put X in my cart", "delete X from cart".
    → Route to `shopping_cart_agent`. (If items must be discovered first, route to `product_qna_agent` first.)

3. EXPLICIT RESERVATION INSTRUCTION — uses warehouse/reservation vocabulary.
    Examples: "reserve these", "hold these for me", "check warehouse stock for X".
    → Route to `warehouse_manager_agent` ONLY AFTER items are confirmed in the cart.

Default rule: if the user's message does NOT contain explicit cart or warehouse vocabulary, do NOT delegate to those agents. Phrases like "can I get", "I want",
"I'd like", "show me", "I need" are PRODUCT QUESTIONS, not purchase instructions. The user must explicitly say "add", "cart", "reserve", "buy", or "order" before those agents are in scope.

## Available Agents

- product_qna_agent: The user is asking a question about a product. This can be a question about available products, their specifications, user reviews etc.
- shopping_cart_agent: The user is asking to add or remove items from the shopping cart or questions about the current shopping cart.
- warehouse_manager_agent: The user is asking to reserve items from the warehouses or about availability of the items in warehouses.
"""

    template = Template(prompt_template)

    prompt = template.render()

    llm = ChatOpenAI(
        model="gpt-5.4-mini",
        reasoning_effort="medium",
        use_responses_api=True
    )
    llm_with_tools = llm.bind_tools(
        [FinalAgentResponse, Plan],
        tool_choice="required"
    )

    response = llm_with_tools.invoke(
        [
            SystemMessage(content=prompt),
            *state.messages
        ]
    )

    final_answer = False
    answer = ""
    # plan = []
    next_agent = ""

    def sanitise_response(response):

        for tool_call in response.tool_calls:
            if tool_call.get("name") == "FinalAgentResponse":
                answer = tool_call.get("args").get("answer")

        return AIMessage(content=answer)

    if len(response.tool_calls) > 0:
        if response.tool_calls[0].get("name") == "Plan":
            # plan = response.tool_calls[0].get("args").get("plan")
            next_agent = response.tool_calls[0].get("args").get("next_agent")
            response = None
        else:
            for tool_call in response.tool_calls:
                if tool_call.get("name") == "FinalAgentResponse":
                    final_answer = True
                    answer = tool_call.get("args").get("answer")

                    response = sanitise_response(response)

    return {
        "messages": [response] if response else [],
        "coordinator_agent": {
            "iteration": state.coordinator_agent.iteration + 1,
            "final_answer": final_answer,
            # "plan": plan,
            "next_agent": next_agent
        },
        "answer": answer
    }

In [ ]:
results = ls_client.evaluate(
    lambda x: coordinator_agent(
        State(
            messages=x["input"]["messages"],
            coordinator_agent=CoordinatorAgentProperties(
                iteration=0,
                final_answer=False,
                plan=[],
                next_agent=""
            ),
            answer=""
        )
    ),
    data="coordinator-delegation-evaluation",
    evaluators=[
        evaluate_coordinator_delegation
    ],
    experiment_prefix="coordinator-delegation",
    max_concurrency=10,
    num_repetitions=5
)

### Coordinator Agent V5 (add worker agents final response and redirect to particular agent in Eval dataset example, since coordinator agent doesn't know from where the response is coming)
Reasoning effort = 'medium'

In [25]:
class Delegation(BaseModel):
    agent: str = Field(description="The agent to delegate the task to.")
    task: str = Field(description="The task to be performed by the agent.")

class Plan(BaseModel):
    next_agent: str = Field(description="The next agent to invoke")
    # plan: List[Delegation] = Field(description="A list of delegations to agents with tasks to be performed in sequence.")

class FinalAgentResponse(BaseModel):

    answer: str = Field(description="Answer to the question")

class CoordinatorAgentProperties(BaseModel):
    iteration: int = 0
    final_answer: bool = False
    # plan: List[Delegation] = []
    next_agent: str = ""
    
class State(BaseModel):
    messages: Annotated[List[Any], add] = []
    coordinator_agent: CoordinatorAgentProperties = CoordinatorAgentProperties()
    answer: str = ""

In [26]:
@traceable(
    name="coordinator_agent",
    run_type="llm",
    metadata={
        "ls_provider": "openai",
        "ls_model_name": "gpt-5.4-mini"
    }
)
def coordinator_agent(state) -> dict:
    
    prompt_template = """You are a Coordinator Agent as part of a shopping assistant.

## Instructions

- Your role is to delegate work to worker agents in order to solve user queries.
- You should output the next agent to invoke.
- Once an agent finishes its task, you will be handed the control back, you should then review the conversation history and output the next delegation via the `Plan` tool.
- If you do not need to delegate work to any agent, output the final answer via the `FinalAgentResponse` tool.
- Do not route to any agent if the user's query needs clarification or is irelevant. Do it yourself.
- Only route to the `warehouse_manager_agent` if the user has confirmed that they want to reserve an item and it was already successfully added to the shopping cart.
- Only route to the `shopping_cart_agent` if the user specifically asks about their shopping cart or wants to add or remove items from the shopping cart.
- Do not delegate work to the same agent twice in a row.

## Routing rules

Match the user's latest message to one of these patterns:

1. PRODUCT QUESTION — asks about products, specs, reviews, availability, suggestions.
    Examples: "can I get some earphones", "show me laptops", "what watches do you have", "tell me about X", "is X any good", "I'd like to see some X".
    → Route ONLY to `product_qna_agent`. When it returns, emit `FinalAgentResponse`. Do NOT chain to cart or warehouse.

2. EXPLICIT CART INSTRUCTION — uses cart vocabulary.
    Examples: "add to my cart", "remove from my cart", "what's in my cart", "put X in my cart", "delete X from cart".
    → Route to `shopping_cart_agent`. (If items must be discovered first, route to `product_qna_agent` first.)

3. EXPLICIT RESERVATION INSTRUCTION — uses warehouse/reservation vocabulary.
    Examples: "reserve these", "hold these for me", "check warehouse stock for X".
    → Route to `warehouse_manager_agent` ONLY AFTER items are confirmed in the cart.

Default rule: if the user's message does NOT contain explicit cart or warehouse vocabulary, do NOT delegate to those agents. Phrases like "can I get", "I want",
"I'd like", "show me", "I need" are PRODUCT QUESTIONS, not purchase instructions. The user must explicitly say "add", "cart", "reserve", "buy", or "order" before those agents are in scope.

## Available Agents

- product_qna_agent: The user is asking a question about a product. This can be a question about available products, their specifications, user reviews etc.
- shopping_cart_agent: The user is asking to add or remove items from the shopping cart or questions about the current shopping cart.
- warehouse_manager_agent: The user is asking to reserve items from the warehouses or about availability of the items in warehouses.
"""

    template = Template(prompt_template)

    prompt = template.render()

    llm = ChatOpenAI(
        model="gpt-5.4-mini",
        reasoning_effort="medium",
        use_responses_api=True
    )
    llm_with_tools = llm.bind_tools(
        [FinalAgentResponse, Plan],
        tool_choice="required"
    )

    response = llm_with_tools.invoke(
        [
            SystemMessage(content=prompt),
            *state.messages
        ]
    )

    final_answer = False
    answer = ""
    # plan = []
    next_agent = ""

    def sanitise_response(response):

        for tool_call in response.tool_calls:
            if tool_call.get("name") == "FinalAgentResponse":
                answer = tool_call.get("args").get("answer")

        return AIMessage(content=answer)

    if len(response.tool_calls) > 0:
        if response.tool_calls[0].get("name") == "Plan":
            # plan = response.tool_calls[0].get("args").get("plan")
            next_agent = response.tool_calls[0].get("args").get("next_agent")
            response = None
        else:
            for tool_call in response.tool_calls:
                if tool_call.get("name") == "FinalAgentResponse":
                    final_answer = True
                    answer = tool_call.get("args").get("answer")

                    response = sanitise_response(response)

    return {
        "messages": [response] if response else [],
        "coordinator_agent": {
            "iteration": state.coordinator_agent.iteration + 1,
            "final_answer": final_answer,
            # "plan": plan,
            "next_agent": next_agent
        },
        "answer": answer
    }

In [ ]:
results = ls_client.evaluate(
    lambda x: coordinator_agent(
        State(
            messages=x["input"]["messages"],
            coordinator_agent=CoordinatorAgentProperties(
                iteration=0,
                final_answer=False,
                plan=[],
                next_agent=""
            ),
            answer=""
        )
    ),
    data="coordinator-delegation-evaluation-2",
    evaluators=[
        evaluate_coordinator_delegation
    ],
    experiment_prefix="coordinator-delegation",
    max_concurrency=10,
    num_repetitions=5
)